# E1 · MRI → doku segmentasyonu — 3B sunum demosu

**Tamamlanan çalışma:** 100 epoch, 44 eğitim / 10 doğrulama, yalnız fold 0.
Bu defter eğitim başlatmaz. Önce gerçek, **önceden kaydedilmiş** doğrulama tahminini gösterir.
İsteğe bağlı son adım, eğitilmiş modeli yükleyip **yeni tahmin** üretir.

- Kayıtlı sonuç gösterimi için Colab'da **CPU** yeterli.
- Canlı GPU tahmini için **L4** seç; ilk hücrede `RUN_LIVE = True` yap, ortam hücresini tekrar çalıştır.
- Kaynak: kendi Drive'ındaki `TMS_E1/runs/E1_L4_001` ve hazırlanmış `handoff_k2_e1.zip`.
- Çıktılar `TMS_E1/demos/` altında yeni klasörlere yazılır. Eğitim dosyaları değiştirilmez.

Gösterilen vakalar doğrulama vakalarıdır; bağımsız test sonucu veya mesh üretimi değildir.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time

EXPERIMENT_ID = "E1_L4_001"
CASE_ID = "GU011"  # Other examples: GU041, NYU005. Validation cases only.
RUN_LIVE = False
DEVICE = "cuda"  # For explicitly enabled inference: "cuda" or "cpu".
USE_TTA = True  # Matches validation; changing it changes the inference protocol.
DRIVE = Path("/content/drive/MyDrive/TMS_E1")
ARCHIVE = DRIVE / "handoff_k2_e1.zip"
EXPERIMENT = DRIVE / "runs" / EXPERIMENT_ID
DATASET = "Dataset001_MultiAxial"
MODEL = EXPERIMENT / DATASET / "nnUNetTrainer_E1Preliminary__nnUNetPlans__3d_fullres"
SAVED = MODEL / "fold_0/validation"
SUMMARY = SAVED / "summary.json"
WORK = Path("/content/e1_demo")
TOOLS = WORK / "tools"
DATA = WORK / "handoff"
VENV = Path("/content/e1-demo-venv")
PYTHON = VENV / "bin/python"
OUTPUTS = DRIVE / "demos" / EXPERIMENT_ID
assert DEVICE in ("cpu", "cuda")
assert EXPERIMENT_ID.replace("_", "").replace("-", "").isalnum()


## 1 · Drive bağlantısı ve kaynaklar
Aynı Google hesabını kullan. Burada yeni GPU tahsisi veya eğitim yapılmaz.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
assert os.path.ismount("/content/drive"), "Drive bağlı değil."
assert ARCHIVE.is_file(), f"Hazırlanmış veri ZIP'i bulunamadı: {ARCHIVE}"
assert MODEL.is_dir() and SUMMARY.is_file(), f"Tamamlanmış model/özet bulunamadı: {MODEL}"
TOOLS.mkdir(parents=True, exist_ok=True)


## 2 · Demo ortamı
Kayıtlı sonuç modu yalnız hafif görüntüleme paketlerini kurar. `RUN_LIVE = True` seçiliyse
ayrıca eğitimle aynı nnU-Net/Torch sürümleri kurulur. İlk kurulum internet ister;
üretilen HTML raporu sonradan internetsiz açılabilir.


In [ ]:
EMBEDDED = {'demo.py': '"""E1 single-MRI inference and saved-result presentation. Never trains or rewrites inputs."""\n\nimport argparse\nimport base64\nimport hashlib\nimport html\nimport json\nimport os\nfrom pathlib import Path\nimport time\nimport zipfile\n\nimport nibabel as nib\nimport numpy as np\n\nLABELS = {"background": 0, "skin": 1, "skull": 2, "csf": 3, "gray_matter": 4, "white_matter": 5}\nTITLES = {1: "Deri", 2: "Kafatası", 3: "BOS / CSF", 4: "Gri cevher", 5: "Beyaz cevher"}\nVALIDATION_CASES = ("GU011", "GU012", "GU018", "GU033", "GU041", "NC012", "NC019", "NC023", "NC029", "NYU005")\n\n\ndef require(condition, message):\n    if not condition:\n        raise ValueError(message)\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef read_volume(path, labels=False):\n    image = nib.load(str(path))\n    require(image.ndim == 3 and 0 < np.prod(image.shape) <= 100_000_000, "Expected a bounded 3D NIfTI volume.")\n    require(np.isfinite(image.affine).all() and abs(np.linalg.det(image.affine[:3, :3])) > 1e-10,\n            "NIfTI affine is invalid.")\n    array = image.get_fdata(dtype=np.float32, caching="unchanged")\n    require(np.isfinite(array).all(), "Volume must contain finite values.")\n    if labels:\n        require(np.all(array == np.round(array)) and np.all((array >= 0) & (array <= 5)), "Expected integer labels 0–5.")\n        array = array.astype(np.uint8)\n        require(np.any(array), "Segmentation is empty.")\n    else:\n        require(np.ptp(array) > 0, "MRI is constant.")\n    return image, array\n\n\ndef compare_volumes(mri, prediction, reference=None):\n    paths = {"mri": Path(mri), "prediction": Path(prediction)}\n    if reference is not None:\n        paths["reference"] = Path(reference)\n        require(paths["reference"].resolve() != paths["prediction"].resolve(), "Reference and prediction are the same file.")\n    loaded = {role: read_volume(path, labels=role != "mri") for role, path in paths.items()}\n    source = loaded["mri"][0]\n    units = {role: image.header.get_xyzt_units()[0] for role, (image, _) in loaded.items()}\n    known_units = set(units.values()) - {"unknown"}\n    require(len(known_units) <= 1, "Known spatial units disagree; no automatic conversion is performed.")\n    for image, _ in loaded.values():\n        require(image.shape == source.shape and np.allclose(image.affine, source.affine, rtol=0, atol=1e-5),\n                "Volume grid differs; no automatic resampling is performed.")\n    warnings = ["Doku isimleri teslim paketindeki geçici eşlemeye dayanır; sonuçlar bağımsız test değildir."]\n    if "unknown" in units.values():\n        warnings.append("Kaynak fiziksel birimleri belirsiz. Bu karşılaştırma voksel düzlemindedir; mm doğruluğunu kanıtlamaz.")\n    per_label = None\n    macro = None\n    if reference is not None:\n        target, guess = loaded["reference"][1], loaded["prediction"][1]\n        confusion = np.bincount((target.astype(np.int16) * 6 + guess).ravel(), minlength=36).reshape(6, 6)\n        per_label = {}\n        for label in range(1, 6):\n            tp = int(confusion[label, label])\n            n_ref, n_pred = int(confusion[label].sum()), int(confusion[:, label].sum())\n            per_label[str(label)] = {"name": TITLES[label], "dice": 2 * tp / (n_ref + n_pred) if n_ref + n_pred else None,\n                                     "reference_voxels": n_ref, "prediction_voxels": n_pred,\n                                     "volume_bias_percent": 100 * (n_pred / n_ref - 1) if n_ref else None}\n        valid = [m["dice"] for m in per_label.values() if m["dice"] is not None]\n        macro = float(np.mean(valid)) if valid else None\n    return {"case_mean_dice": macro, "per_label": per_label, "grid_shape": list(source.shape),\n            "axis_codes": list(nib.aff2axcodes(source.affine)), "spatial_units": units, "warnings": warnings,\n            "source_files": {role: {"path": str(path.resolve()), "sha256": sha256(path)} for role, path in paths.items()}}\n\n\ndef fresh_directory(path):\n    path = Path(path).resolve()\n    require(not path.exists(), f"Output already exists; choose a fresh directory: {path}")\n    require(not any((parent / ".git").exists() for parent in (path, *path.parents)), "Keep MRI demo outputs outside Git.")\n    path.mkdir(parents=True)\n    return path\n\n\ndef render_comparison(mri, prediction, reference, destination):\n    os.environ.setdefault("MPLBACKEND", "Agg")\n    os.environ.setdefault("MPLCONFIGDIR", str(Path(destination).parent / ".matplotlib"))\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n    from matplotlib.colors import ListedColormap\n    from matplotlib.patches import Patch\n\n    _, image = read_volume(mri)\n    _, guess = read_volume(prediction, labels=True)\n    target = read_volume(reference, labels=True)[1] if reference else None\n    focus = target if target is not None else guess\n    coordinates = np.where(np.isin(focus, [3, 4, 5]))\n    if not coordinates[0].size:\n        coordinates = np.where(focus > 0)\n    center = [int(np.median(c)) for c in coordinates]\n    colors = [(0, 0, 0, 0), (.99, .65, .52, .5), (1, .9, .4, .55), (.25, .55, 1, .6),\n              (.65, .25, .82, .6), (.93, .93, .93, .6)]\n    foreground = image[image != 0]\n    low, high = np.percentile(foreground, [.5, 99.5])\n    columns = 4 if target is not None else 2\n    fig, axes = plt.subplots(3, columns, figsize=(columns * 3.1, 9.2), squeeze=False)\n    for row, (axis, title) in enumerate(((2, "Aksiyel"), (1, "Koronal"), (0, "Sagittal"))):\n        plane = np.take(image, center[axis], axis).T\n        masks = [None, np.take(guess, center[axis], axis).T]\n        if target is not None:\n            truth = np.take(target, center[axis], axis).T\n            masks = [None, truth, masks[1], (truth != masks[1]).astype(np.uint8)]\n        titles = ["MRI", "Model tahmini"] if target is None else ["MRI", "Referans etiket", "Model tahmini", "Etiket farkı"]\n        for column, mask in enumerate(masks):\n            ax = axes[row, column]\n            ax.imshow(plane, cmap="gray", vmin=low, vmax=high, origin="lower", interpolation="nearest")\n            if mask is not None:\n                cmap = ListedColormap([(0, 0, 0, 0), (1, .45, 0, .85)]) if column == 3 else ListedColormap(colors)\n                ax.imshow(mask, cmap=cmap, vmin=0, vmax=1 if column == 3 else 5, origin="lower", interpolation="nearest")\n            ax.set_title(f"{title} · voksel {center[axis]}\\n{titles[column]}", fontsize=10)\n            ax.axis("off")\n    fig.legend(handles=[Patch(color=colors[i][:3], label=TITLES[i]) for i in range(1, 6)], loc="lower center", ncol=5)\n    fig.suptitle("E1 · MRI ve doku segmentasyonu\\nAynı voksel ızgarası; sınırlı kesit incelemesi", fontsize=13)\n    fig.tight_layout(rect=(0, .04, 1, .94))\n    fig.savefig(destination, dpi=140)\n    plt.close(fig)\n\n\ndef cohort_metrics(path):\n    if path is None:\n        return None\n    data = json.loads(Path(path).read_text())\n    rows = data["metric_per_case"]\n    ids = [Path(row["reference_file"]).name.removesuffix(".nii.gz") for row in rows]\n    require(len(ids) == 10 and set(ids) == set(VALIDATION_CASES), "Expected the fixed ten validation cases.")\n    means = {str(i): float(np.mean([row["metrics"][str(i)]["Dice"] for row in rows])) for i in range(1, 6)}\n    require(all(np.isfinite(x) and 0 <= x <= 1 for x in means.values()), "Invalid validation Dice.")\n    value = float(np.mean(list(means.values())))\n    require(np.isclose(value, data["foreground_mean"]["Dice"], rtol=0, atol=1e-10), "Validation summary mean disagrees.")\n    return {"validation_cases": 10, "mean_dice": value, "per_label_dice": means, "summary_sha256": sha256(path)}\n\n\ndef make_report(mri, prediction, output, reference=None, mode="recorded", timing=None, summary=None, model=None):\n    require(mode in ("recorded", "inference"), "Unknown demo mode.")\n    require(mode != "recorded" or timing is None, "Recorded predictions cannot claim live inference timing.")\n    require(mode != "inference" or timing is not None, "New inference requires measured timing.")\n    result = compare_volumes(mri, prediction, reference)\n    result.update({"mode": mode, "timing": timing, "cohort": cohort_metrics(summary), "model": model,\n                   "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "held_out_test": False})\n    folder = fresh_directory(output)\n    render_comparison(mri, prediction, reference, folder / "comparison.png")\n    from surface_view import make_view\n    source, _ = read_volume(mri)\n    _, labels = read_volume(prediction, labels=True)\n    surface_html, surface_metadata = make_view(labels, source.affine, TITLES)\n    result["surface_view"] = surface_metadata\n    (folder / "report.json").write_text(json.dumps(result, indent=2, allow_nan=False) + "\\n")\n    number = lambda value: "—" if value is None else f"{value:.4f}"\n    rows = "".join(f"<tr><td>{html.escape(m[\'name\'])}</td><td>{number(m[\'dice\'])}</td></tr>" for m in (result["per_label"] or {}).values())\n    cohort = result["cohort"]\n    cohort_text = f"Eğitim sonunda kaydedilen 10 vakalık doğrulama · ortalama Dice <strong>{cohort[\'mean_dice\']:.4f}</strong>" if cohort else "Bu raporda kohort skoru sunulmuyor."\n    score_table = (f"<table><tr><th>Doku</th><th>Bu vaka — Dice</th></tr>{rows}</table>"\n                   if rows else "<p>Referans yok; bu MRI için Dice hesaplanmadı.</p>")\n    difference_legend = "<p>Turuncu: referans ve tahminin farklı etiket verdiği vokseller.</p>" if reference else ""\n    badge = "Kaydedilmiş tahmin · canlı çalıştırma değil" if mode == "recorded" else f"Yeni tahmin · {html.escape(timing[\'device\'].upper())}"\n    timing_text = "" if timing is None else (f"<p>Model yükleme: {timing[\'model_load_seconds\']:.1f} sn · "\n        f"Ön işleme + tahmin + NIfTI kaydı: {timing[\'prediction_seconds\']:.1f} sn. "\n        f"Ayna çevirmeli tahmin: {\'açık\' if timing[\'tta\'] else \'kapalı\'}.</p>")\n    picture = base64.b64encode((folder / "comparison.png").read_bytes()).decode()\n    notes = "".join(f"<li>{html.escape(note)}</li>" for note in result["warnings"])\n    page = f\'\'\'<!doctype html><html lang="tr"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">\n<title>E1 · MRI doku segmentasyonu</title><style>\nbody{{font:17px/1.55 system-ui,sans-serif;color:#18232d;background:#f3f5f7;margin:0}}main{{max-width:1180px;margin:36px auto;padding:28px;background:white;border-radius:14px}}\nh1{{font-size:32px;margin:10px 0}}.badge{{display:inline-block;background:#e7f0ff;color:#174b87;padding:6px 12px;border-radius:6px}}img{{width:100%;height:auto}}table{{border-collapse:collapse;min-width:320px}}td,th{{text-align:left;padding:9px 18px;border-bottom:1px solid #dde3e8}}small{{color:#53606c}}\n</style><main><div class="badge">{badge}</div><h1>MRI → doku segmentasyonu</h1>\n<p>E1 ön model · 100 epoch · 44 eğitim / 10 doğrulama · fold 0</p><p>{cohort_text}</p>\n<p>Seçilen vaka: <strong>{html.escape(Path(mri).name)}</strong> · vaka ortalama Dice: <strong>{number(result[\'case_mean_dice\'])}</strong></p>\n<p><small>Rapor kayıt zamanı: {result[\'created_utc\']} (UTC). Bu dosyayı yeniden açmak yeni tahmin başlatmaz.</small></p>\n{timing_text}{surface_html}{score_table}\n<img alt="MRI, referans etiket ve model tahmininin eşleşen üç düzlemde karşılaştırması" src="data:image/png;base64,{picture}">\n{difference_legend}<ul>{notes}</ul>\n<small>Bu çıktı doku segmentasyonu ve gösterim yüzeyleridir; simülasyona hazır hacim mesh\'i veya klinik doğrulama değildir. Kaydedilmiş rapor internet bağlantısı gerektirmez.</small></main></html>\'\'\'\n    (folder / "report.html").write_text(page)\n    return folder / "report.html"\n\n\ndef validate_model(folder):\n    folder = Path(folder)\n    require((folder / "dataset.json").is_file(), "Model dataset.json is missing.")\n    dataset = json.loads((folder / "dataset.json").read_text())\n    require(dataset.get("labels") == LABELS, "Model labels differ from the E1 mapping.")\n    require(dataset.get("channel_names") == {"0": "T1"} and dataset.get("file_ending") == ".nii.gz", "Expected the E1 T1 model.")\n    require((folder / "plans.json").is_file(), "Model plans.json is missing.")\n    plans = json.loads((folder / "plans.json").read_text())\n    require("3d_fullres" in plans.get("configurations", {}), "Model lacks 3d_fullres.")\n    checkpoint = folder / "fold_0/checkpoint_final.pth"\n    require(checkpoint.is_file() and zipfile.is_zipfile(checkpoint), "Final checkpoint is missing or incomplete.")\n    with zipfile.ZipFile(checkpoint) as archive:\n        require(archive.testzip() is None, "Final checkpoint integrity check failed.")\n    return {"checkpoint_sha256": sha256(checkpoint), "plans_sha256": sha256(folder / "plans.json"),\n            "dataset_sha256": sha256(folder / "dataset.json"), "fold": 0, "checkpoint": "checkpoint_final.pth"}\n\n\ndef register_trainer():\n    import nnunetv2\n    source = Path(__file__).with_name("trainers.py").read_text()\n    target = Path(nnunetv2.__file__).parent / "training/nnUNetTrainer/variants/training_length/nnUNetTrainer_E1.py"\n    if target.exists():\n        require(target.read_text() == source, "Installed E1 trainer differs; use the reviewed isolated environment.")\n    else:\n        target.write_text(source)\n\n\ndef predict(model_folder, mri, output, reference=None, device="cpu", tta=True, summary=None):\n    require(device in ("cpu", "cuda"), "Use cpu or cuda.")\n    require(not Path(output).resolve().is_relative_to(Path(model_folder).resolve()),\n            "Output must be outside the model directory.")\n    read_volume(mri)\n    model = validate_model(model_folder)\n    folder = fresh_directory(output)\n    status = folder / "status.json"\n    status.write_text(json.dumps({"state": "running", "mode": "inference"}) + "\\n")\n    try:\n        import importlib.metadata\n        import torch\n        require(importlib.metadata.version("nnunetv2") == "2.8.1", "Use nnunetv2 2.8.1.")\n        require(torch.__version__.split("+")[0] == "2.6.0", "Use torch 2.6.0.")\n        require(device != "cuda" or torch.cuda.is_available(), "CUDA unavailable; select a GPU runtime.")\n        os.environ["nnUNet_compile"] = "false"\n        register_trainer()\n        from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor\n        sync = lambda: torch.cuda.synchronize() if device == "cuda" else None\n        sync()\n        start = time.perf_counter()\n        predictor = nnUNetPredictor(tile_step_size=.5, use_gaussian=True, use_mirroring=tta,\n                                   device=torch.device(device), perform_everything_on_device=device == "cuda",\n                                   verbose=False, verbose_preprocessing=False, allow_tqdm=True)\n        predictor.initialize_from_trained_model_folder(str(model_folder), use_folds=(0,), checkpoint_name="checkpoint_final.pth")\n        require(predictor.trainer_name == "nnUNetTrainer_E1Preliminary", "Expected the completed 100-epoch E1 model.")\n        sync()\n        loaded = time.perf_counter()\n        destination = folder / "prediction"\n        predictor.predict_from_files([[str(Path(mri).resolve())]], [str(destination)], save_probabilities=False,\n                                     overwrite=False, num_processes_preprocessing=1, num_processes_segmentation_export=1)\n        sync()\n        finished = time.perf_counter()\n        timing = {"device": device, "model_load_seconds": loaded - start, "prediction_seconds": finished - loaded,\n                  "tta": tta, "torch": torch.__version__, "nnunetv2": "2.8.1",\n                  "hardware": torch.cuda.get_device_name(0) if device == "cuda" else __import__("platform").platform()}\n        report = make_report(mri, destination.with_suffix(".nii.gz"), folder / "report", reference=reference,\n                             mode="inference", timing=timing, summary=summary, model=model)\n        status.write_text(json.dumps({"state": "completed", "report": str(report), "timing": timing}, indent=2) + "\\n")\n        return report\n    except BaseException as error:\n        status.write_text(json.dumps({"state": "failed", "error": str(error)}, indent=2) + "\\n")\n        raise\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    commands = parser.add_subparsers(dest="command", required=True)\n    for name in ("report", "predict"):\n        command = commands.add_parser(name)\n        command.add_argument("--mri", type=Path, required=True)\n        command.add_argument("--reference", type=Path)\n        command.add_argument("--summary", type=Path)\n        command.add_argument("--out", type=Path, required=True)\n        if name == "report":\n            command.add_argument("--recorded-prediction", type=Path, required=True)\n        else:\n            command.add_argument("--model", type=Path, required=True)\n            command.add_argument("--device", choices=("cpu", "cuda"), default="cpu")\n            command.add_argument("--disable-tta", action="store_true")\n    args = parser.parse_args()\n    if args.command == "report":\n        report = make_report(args.mri, args.recorded_prediction, args.out, reference=args.reference, summary=args.summary)\n    else:\n        report = predict(args.model, args.mri, args.out, reference=args.reference, device=args.device,\n                         tta=not args.disable_tta, summary=args.summary)\n    print(f"Rapor hazır: {report}", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'surface_view.py': '"""Approximate display surfaces from tissue labels; no FEM mesh or label editing."""\n\nimport html\nfrom pathlib import Path\n\nimport nibabel as nib\nimport numpy as np\n\nCOLORS = {1: "#fca685", 2: "#ead675", 3: "#408cff", 4: "#ad60cf", 5: "#e4e7ed"}\nOPACITY = {1: .12, 2: .3, 3: .25, 4: 1., 5: 1.}\n\n\ndef extract_surfaces(labels, affine, step_size=2, max_faces=360_000):\n    from skimage.measure import marching_cubes\n\n    labels, affine = np.asarray(labels), np.asarray(affine)\n    if labels.ndim != 3 or not 0 < labels.size <= 100_000_000 or not np.isfinite(labels).all():\n        raise ValueError("Expected bounded finite 3D labels.")\n    if not np.all((labels >= 0) & (labels <= 5) & (labels == np.round(labels))):\n        raise ValueError("Expected integer labels 0–5.")\n    if affine.shape != (4, 4) or not np.isfinite(affine).all() or abs(np.linalg.det(affine[:3, :3])) < 1e-10:\n        raise ValueError("Invalid source affine.")\n    if not 1 <= step_size <= 4 or max_faces < 1 or not np.any(labels):\n        raise ValueError("Invalid surface extraction settings or empty labels.")\n    ids = [int(value) for value in np.unique(labels) if value]\n    # shortcut: coarser marching-cubes sampling is for display only; use validated full-resolution meshing for simulation.\n    for step in range(step_size, 5):\n        surfaces = []\n        for label in ids:\n            mask = labels == label\n            positions = np.where(mask)\n            lower = np.array([int(p.min()) for p in positions])\n            upper = np.array([int(p.max()) + 1 for p in positions])\n            cropped = mask[tuple(slice(a, b) for a, b in zip(lower, upper))]\n            used = step\n            padded = np.pad(cropped, used)\n            try:\n                vertices, faces, _, _ = marching_cubes(padded, .5, step_size=used, allow_degenerate=False)\n            except (ValueError, RuntimeError):\n                used = 1\n                vertices, faces, _, _ = marching_cubes(np.pad(cropped, 1), .5, step_size=1, allow_degenerate=False)\n            vertices = nib.affines.apply_affine(affine, vertices + lower - used).astype(np.float32)\n            surfaces.append({"label": label, "vertices": vertices, "faces": faces.astype(np.int32), "step_size": used})\n        if sum(len(mesh["faces"]) for mesh in surfaces) <= max_faces:\n            return surfaces\n    raise ValueError("Surface geometry is too complex for the bounded browser preview.")\n\n\ndef make_view(labels, affine, names):\n    import plotly.graph_objects as go\n\n    surfaces = extract_surfaces(labels, affine)\n    figure = go.Figure()\n    controls = []\n    for index, mesh in enumerate(surfaces):\n        label, points, faces = mesh["label"], mesh["vertices"], mesh["faces"]\n        visible = label in (1, 4)\n        alpha = OPACITY[label]\n        name = html.escape(names[label])\n        figure.add_trace(go.Mesh3d(x=points[:, 0], y=points[:, 1], z=points[:, 2],\n                                  i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],\n                                  color=COLORS[label], opacity=alpha, visible=visible, name=names[label],\n                                  showlegend=False, hoverinfo="name", flatshading=False,\n                                  lighting={"ambient": .6, "diffuse": .75, "specular": .08, "roughness": .75},\n                                  lightposition={"x": 100, "y": 200, "z": 400}))\n        controls.append(f\'\'\'<div class="surface-row"><label>\n<input type="checkbox" data-surface-index="{index}" data-surface-label="{label}" {\'checked\' if visible else \'\'} disabled>\n<span style="color:{COLORS[label]}">●</span> {name}</label>\n<label class="opacity-label">Saydamlık / görünürlük\n<input aria-label="{name} görünürlük oranı" type="range" min="0" max="100" value="{round(alpha * 100)}" data-opacity-index="{index}" disabled></label></div>\'\'\')\n    figure.update_layout(height=620, margin={"l": 0, "r": 0, "t": 5, "b": 0}, paper_bgcolor="#f4f6f9",\n                         scene={"aspectmode": "data", "dragmode": "orbit", "bgcolor": "#f4f6f9",\n                                "xaxis": {"title": "R ↔ L", "visible": False},\n                                "yaxis": {"title": "A ↔ P", "visible": False},\n                                "zaxis": {"title": "S ↔ I", "visible": False},\n                                "camera": {"up": {"x": 0, "y": 0, "z": 1}, "eye": {"x": 1.6, "y": -1.8, "z": 1.1}}})\n    script = Path(__file__).with_name("surface_controls.js").read_text()\n    plot = figure.to_html(full_html=False, include_plotlyjs=True, include_mathjax=False, div_id="e1-surface-plot",\n                          config={"responsive": True, "displaylogo": False, "scrollZoom": True}, post_script=script)\n    steps = sorted({mesh["step_size"] for mesh in surfaces})\n    buttons = "".join(f\'<button type="button" data-surface-preset="{key}" disabled>{title}</button>\'\n                      for key, title in (("brain", "Beyin"), ("head", "Kafa"), ("all", "Tümü"), ("reset", "Sıfırla")))\n    fragment = f\'\'\'<section aria-label="Üç boyutlu segmentasyon yüzeyleri"><h2>3B doku yüzeyleri</h2>\n<p>Sürükle: döndür · tekerlek: yakınlaştır · dokuları kutucuklardan seç. Sıfır görünürlük tamamen saydamdır.</p>\n<style>.surface-layout{{display:grid;grid-template-columns:210px minmax(0,1fr);gap:12px;background:#f4f6f9;border-radius:12px;padding:12px}}.surface-row{{padding:12px 0;border-bottom:1px solid #dce1e8}}.surface-row label{{display:block}}.opacity-label{{font-size:12px;color:#53606c}}.surface-row input[type=range]{{width:100%}}.surface-buttons{{display:flex;flex-wrap:wrap;gap:6px}}.surface-buttons button{{padding:7px 10px;background:white;border:1px solid #cbd3dd;border-radius:6px;cursor:pointer}}@media(max-width:700px){{.surface-layout{{grid-template-columns:1fr}}}}</style>\n<div class="surface-layout"><aside><div class="surface-buttons">{buttons}</div>{\'\'.join(controls)}\n<small id="e1-surface-status" role="status">3B görünüm hazırlanıyor. Görünmezse indirilen HTML raporunu Chrome\'da aç.</small></aside>{plot}</div>\n<p><small>Gösterim için yaklaşık yüzeyler · örnekleme adımı: {\', \'.join(map(str, steps))} voksel.\nKoordinatlar kaynak NIfTI affine\'inden alınır; fiziksel ölçek ayrıca doğrulanmalıdır.\nBu görünüm SimNIBS/FEM hacim mesh\'i değildir. Skorlar sadeleştirilmiş yüzeylerden değil, özgün etiket hacminden hesaplanır.</small></p></section>\'\'\'\n    metadata = {"visualization_only": True, "fem_mesh": False, "labels": [mesh["label"] for mesh in surfaces],\n                "triangles": sum(len(mesh["faces"]) for mesh in surfaces),\n                "vertices": sum(len(mesh["vertices"]) for mesh in surfaces), "step_sizes": steps,\n                "source_affine_preserved": True, "physical_scale_validated": False}\n    return fragment, metadata\n', 'surface_controls.js': "const graph = document.getElementById('e1-surface-plot');\nconst status = document.getElementById('e1-surface-status');\nconst checks = Array.from(document.querySelectorAll('[data-surface-index]'));\nconst ranges = Array.from(document.querySelectorAll('[data-opacity-index]'));\nconst defaults = {1: 12, 2: 30, 3: 25, 4: 100, 5: 100};\n\nchecks.forEach(check => {\n  check.disabled = false;\n  check.addEventListener('change', () => {\n    Plotly.restyle(graph, {visible: check.checked}, [Number(check.dataset.surfaceIndex)]);\n  });\n});\nranges.forEach(range => {\n  range.disabled = false;\n  range.addEventListener('input', () => {\n    Plotly.restyle(graph, {opacity: Number(range.value) / 100}, [Number(range.dataset.opacityIndex)]);\n  });\n});\ndocument.querySelectorAll('[data-surface-preset]').forEach(button => {\n  button.disabled = false;\n  button.addEventListener('click', () => {\n    const preset = button.dataset.surfacePreset;\n    checks.forEach((check, index) => {\n      const label = Number(check.dataset.surfaceLabel);\n      let visible = label === 1 || label === 4;\n      let opacity = defaults[label];\n      if (preset === 'brain') visible = label === 4 || label === 5;\n      if (preset === 'head') {visible = label === 1; opacity = 100;}\n      if (preset === 'all') {visible = true; if (label === 4) opacity = 60;}\n      check.checked = visible;\n      ranges[index].value = String(opacity);\n      Plotly.restyle(graph, {visible, opacity: opacity / 100}, [index]);\n    });\n    if (preset === 'reset') {\n      Plotly.relayout(graph, {'scene.camera': {up: {x: 0, y: 0, z: 1}, eye: {x: 1.6, y: -1.8, z: 1.1}}});\n    }\n  });\n});\nstatus.textContent = '3B hazır · görünürlük kaydırıcısı: 0 saydam, 100 opak.';\ngraph.on('plotly_webglcontextlost', () => {\n  status.textContent = 'Tarayıcının 3B bağlamı kesildi. Raporu yeniden aç veya 2B kesitleri kullan.';\n});\n", 'trainers.py': '"""nnU-Net 2.8.1 E1 variants: seed initialization and save each epoch."""\n\nimport random\nimport numpy as np\nimport torch\n\nfrom nnunetv2.training.nnUNetTrainer.variants.training_length.nnUNetTrainer_Xepochs import (\n    nnUNetTrainer_5epochs, nnUNetTrainer_100epochs,\n)\n\n\nclass nnUNetTrainer_E1Smoke(nnUNetTrainer_5epochs):\n    def __init__(self, plans, configuration, fold, dataset_json, device=torch.device("cuda")):\n        random.seed(42)\n        np.random.seed(42)\n        torch.manual_seed(42)\n        super().__init__(plans, configuration, fold, dataset_json, device)\n        self.save_every = 1\n\n\nclass nnUNetTrainer_E1Preliminary(nnUNetTrainer_100epochs):\n    def __init__(self, plans, configuration, fold, dataset_json, device=torch.device("cuda")):\n        random.seed(42)\n        np.random.seed(42)\n        torch.manual_seed(42)\n        super().__init__(plans, configuration, fold, dataset_json, device)\n        self.save_every = 1\n', 'colab_helpers.py': '"""Small reusable Colab helpers; importing this module never starts training."""\n\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path, PurePosixPath\nimport shutil\nimport signal\nimport stat\nimport subprocess\nimport tempfile\nimport time\nimport zipfile\n\nDATASET = "Dataset001_MultiAxial"\n\n\ndef require(condition, message):\n    if not condition:\n        raise ValueError(message)\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef gpu_probe():\n    import importlib.metadata\n    import torch\n    require(importlib.metadata.version("nnunetv2") == "2.8.1", "Expected nnU-Net 2.8.1.")\n    require(torch.__version__.split("+")[0] == "2.6.0" and torch.version.cuda == "12.4", "Expected torch 2.6.0 / CUDA 12.4.")\n    require(torch.cuda.is_available(), "Choose a GPU runtime in Colab; CUDA is unavailable.")\n    props = torch.cuda.get_device_properties(0)\n    require(props.total_memory >= 12 * 1024**3, "This handoff requests a GPU with at least 12 GB VRAM.")\n    layer = torch.nn.Conv3d(1, 2, 3).cuda()\n    value = torch.randn(1, 1, 12, 12, 12, device="cuda")\n    with torch.autocast("cuda"):\n        result = layer(value).square().mean()\n    result.backward()\n    torch.cuda.synchronize()\n    del layer, value, result\n    torch.cuda.empty_cache()\n    return {"gpu": props.name, "vram_gib": props.total_memory / 1024**3,\n            "torch": torch.__version__, "cuda": torch.version.cuda,\n            "nnunetv2": importlib.metadata.version("nnunetv2"), "conv3d_backward_passed": True}\n\n\ndef pack_handoff(root, report, destination):\n    root, destination = Path(root).resolve(), Path(destination).resolve()\n    require(not destination.exists(), "Archive already exists; choose a new output filename.")\n    require(not any((parent / ".git").exists() for parent in destination.parents), "Keep MRI archives outside Git.")\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    manifest = {"files": report["files"], "dataset_sha256": report["dataset_sha256"]}\n    partial = destination.with_suffix(destination.suffix + ".partial")\n    try:\n        with zipfile.ZipFile(partial, "x", compression=zipfile.ZIP_STORED) as archive:\n            for relative, digest in sorted(manifest["files"].items()):\n                source = root / relative\n                require(sha256(source) == digest, f"Input changed after audit: {relative}")\n                archive.write(source, relative)\n            archive.writestr("expected_files.json", json.dumps(manifest, indent=2) + "\\n")\n        partial.rename(destination)\n    except BaseException:\n        partial.unlink(missing_ok=True)\n        raise\n    return sha256(destination)\n\n\ndef extract_handoff(archive_path, destination):\n    archive_path, destination = Path(archive_path), Path(destination).resolve()\n    archive_digest = sha256(archive_path)\n    marker = destination / ".archive_sha256"\n    if destination.exists():\n        require(marker.is_file() and marker.read_text().strip() == archive_digest,\n                "Existing extraction belongs to another archive. Choose a fresh destination.")\n        return destination\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(archive_path) as archive:\n        names = archive.namelist()\n        require(len(names) == len(set(names)), "Archive has duplicate paths.")\n        require("expected_files.json" in names and archive.getinfo("expected_files.json").file_size < 2_000_000,\n                "Use the prepared handoff_k2_e1.zip with its checksum manifest.")\n        manifest = json.loads(archive.read("expected_files.json"))\n        expected = manifest["files"]\n        require(set(names) == set(expected) | {"expected_files.json"}, "Archive contains unexpected or missing entries.")\n        require(sum(info.file_size for info in archive.infolist()) <= 20_000_000_000, "Archive is too large.")\n        for info in archive.infolist():\n            path = PurePosixPath(info.filename)\n            require(not path.is_absolute() and ".." not in path.parts and "\\\\" not in info.filename,\n                    "Unsafe archive path.")\n            require(not stat.S_ISLNK(info.external_attr >> 16) and not info.is_dir(), "Archive links/directories are not supported.")\n            allowed = (info.filename in ("splits_final.json", "expected_files.json", f"nnUNet_raw/{DATASET}/dataset.json")\n                       or (path.parent.as_posix() in (f"nnUNet_raw/{DATASET}/imagesTr", f"nnUNet_raw/{DATASET}/labelsTr")\n                           and path.name.endswith(".nii.gz")))\n            require(allowed, "Archive contains files outside the E1 dataset.")\n        with tempfile.TemporaryDirectory(prefix="e1-extract-", dir=destination.parent) as folder:\n            temporary = Path(folder)\n            for info in archive.infolist():\n                output = temporary / info.filename\n                output.parent.mkdir(parents=True, exist_ok=True)\n                with archive.open(info) as source, output.open("xb") as target:\n                    shutil.copyfileobj(source, target, length=1024 * 1024)\n                if info.filename in expected:\n                    require(sha256(output) == expected[info.filename], f"Archive checksum mismatch: {info.filename}")\n            (temporary / ".archive_sha256").write_text(archive_digest + "\\n")\n            temporary.rename(destination)\n    return destination\n\n\ndef verify_manifest(root, report):\n    manifest = json.loads((Path(root) / "expected_files.json").read_text())\n    require(report["files"] == manifest["files"] and report["dataset_sha256"] == manifest["dataset_sha256"],\n            "Extracted data differs from the audited handoff manifest.")\n\n\ndef install_split(root, preprocessed):\n    source = Path(root) / "splits_final.json"\n    target = Path(preprocessed) / DATASET / "splits_final.json"\n    target.parent.mkdir(parents=True, exist_ok=True)\n    if target.exists():\n        require(json.loads(target.read_text()) == json.loads(source.read_text()), "Existing split differs; refusing to overwrite it.")\n    else:\n        shutil.copyfile(source, target)\n    return sha256(target)\n\n\ndef training_command(executable, trainer, fold_directory, resume=False):\n    require(trainer in ("nnUNetTrainer_E1Smoke", "nnUNetTrainer_E1Preliminary"), "Unexpected E1 trainer.")\n    folder = Path(fold_directory)\n    if resume:\n        checkpoints = [folder / f"checkpoint_{name}.pth" for name in ("final", "latest", "best")]\n        checkpoint = next((path for path in checkpoints if path.is_file()), None)\n        require(checkpoint is not None, "Resume requested but no checkpoint exists; refusing to silently restart.")\n        require(not (folder / "validation/summary.json").is_file(), "Run already has completed validation; inspect results or use a new experiment ID.")\n        require(zipfile.is_zipfile(checkpoint), "Checkpoint is incomplete or uses an unexpected format.")\n        with zipfile.ZipFile(checkpoint) as archive:\n            require(archive.testzip() is None, "Checkpoint archive failed its integrity check.")\n    else:\n        require(not folder.exists(), "Run directory exists. Use resume or a new experiment ID; no overwrite allowed.")\n    return [str(executable), "1", "3d_fullres", "0", "-tr", trainer, "-device", "cuda"] + (["--c"] if resume else [])\n\n\ndef record_run(path, contract):\n    path = Path(path)\n    if path.exists():\n        require(json.loads(path.read_text()) == contract, "Experiment contract changed; use a new experiment ID.")\n    else:\n        path.parent.mkdir(parents=True, exist_ok=True)\n        path.write_text(json.dumps(contract, indent=2) + "\\n")\n\n\ndef run_logged(command, log_path, env=None):\n    """Tail a durable log; stop owned process group when the notebook is interrupted."""\n    log_path = Path(log_path)\n    log_path.parent.mkdir(parents=True, exist_ok=True)\n    started = time.monotonic()\n    child_env = dict(os.environ if env is None else env)\n    child_env["PYTHONUNBUFFERED"] = "1"\n    with log_path.open("x", buffering=1) as output:\n        process = subprocess.Popen(command, stdout=output, stderr=subprocess.STDOUT,\n                                   stdin=subprocess.DEVNULL, env=child_env, start_new_session=True)\n        try:\n            with log_path.open(errors="replace") as reader:\n                while True:\n                    text = reader.read()\n                    if text:\n                        print(text, end="", flush=True)\n                    code = process.poll()\n                    if code is not None:\n                        print(reader.read(), end="", flush=True)\n                        if code != 0:\n                            raise subprocess.CalledProcessError(code, command)\n                        return {"elapsed_seconds": time.monotonic() - started, "exit_code": code}\n                    time.sleep(.3)\n        finally:\n            # Colab is Linux; group ownership prevents orphaned augmentation workers.\n            try:\n                os.killpg(process.pid, signal.SIGTERM)\n            except ProcessLookupError:\n                pass\n            if process.poll() is None:\n                try:\n                    process.wait(timeout=10)\n                except subprocess.TimeoutExpired:\n                    os.killpg(process.pid, signal.SIGKILL)\n                    process.wait()\n            try:\n                os.killpg(process.pid, signal.SIGKILL)\n            except ProcessLookupError:\n                pass\n', 'requirements-colab.lock.txt': '# This file was autogenerated by uv via the following command:\n#    uv pip compile tools/e1_training/requirements-colab.txt --python-version 3.11 --python-platform x86_64-unknown-linux-gnu --torch-backend cu124 --output-file /private/tmp/e1-linux-requirements.txt\nacvl-utils==0.2.7\n    # via nnunetv2\nannotated-types==0.8.0\n    # via pydantic\nanyio==4.16.0\n    # via\n    #   httpx\n    #   httpx2\nbatchgenerators==0.25.6\n    # via\n    #   acvl-utils\n    #   batchgeneratorsv2\n    #   nnunetv2\nbatchgeneratorsv2==0.3.7\n    # via nnunetv2\nblosc2==4.14.1\n    # via\n    #   acvl-utils\n    #   nnunetv2\ncertifi==2026.10.23\n    # via\n    #   httpcore\n    #   httpx\n    #   requests\ncharset-normalizer==3.5.2\n    # via requests\nclick==8.5.0\n    # via huggingface-hub\ncloudpickle==3.1.2\n    # via joblib\nconnected-components-3d==4.1.1\n    # via acvl-utils\ncontourpy==1.3.3\n    # via matplotlib\ncycler==0.12.1\n    # via matplotlib\ndynamic-network-architectures==0.4.4\n    # via nnunetv2\neinops==0.8.2\n    # via\n    #   dynamic-network-architectures\n    #   nnunetv2\nfilelock==4.2.7\n    # via\n    #   huggingface-hub\n    #   torch\nfonttools==4.68.0\n    # via matplotlib\nfsspec==2026.9.0\n    # via\n    #   huggingface-hub\n    #   torch\ngraphviz==0.21\n    # via nnunetv2\nh11==0.16.0\n    # via\n    #   httpcore\n    #   httpcore2\nh2==4.4.1\n    # via httpx\nhf-xet==1.8.0\n    # via huggingface-hub\nhpack==4.2.0\n    # via h2\nhttpcore==1.0.9\n    # via httpx\nhttpcore2==2.16.0\n    # via httpx2\nhttpx==0.28.1\n    # via blosc2\nhttpx2==2.16.0\n    # via huggingface-hub\nhuggingface-hub==2.4.0\n    # via timm\nhyperframe==6.1.0\n    # via h2\nidna==3.20\n    # via\n    #   anyio\n    #   httpx\n    #   httpx2\n    #   requests\nimagecodecs==2026.1.14\n    # via nnunetv2\nimageio==2.38.1\n    # via scikit-image\nimportlib-resources==7.1.0\n    # via nibabel\njinja2==3.1.6\n    # via torch\njoblib==1.6.0\n    # via scikit-learn\nkiwisolver==1.5.1\n    # via matplotlib\nlazy-loader==0.6\n    # via scikit-image\nmarkdown-it-py==4.2.0\n    # via rich\nmarkupsafe==3.0.4\n    # via jinja2\nmatplotlib==3.11.2\n    # via\n    #   nnunetv2\n    #   seaborn\nmdurl==0.1.2\n    # via markdown-it-py\nmpmath==1.3.0\n    # via sympy\nmsgpack==1.2.4\n    # via blosc2\nnarwhals==2.28.0\n    # via scikit-learn\nndindex==1.10.1\n    # via blosc2\nnetworkx==3.6.1\n    # via\n    #   scikit-image\n    #   torch\nnibabel==5.4.2\n    # via\n    #   -r tools/e1_training/requirements-colab.txt\n    #   nnunetv2\nnnunetv2==2.8.1\n    # via -r tools/e1_training/requirements-colab.txt\nnumexpr==2.14.2\n    # via blosc2\nnumpy==1.26.4\n    # via\n    #   -r tools/e1_training/requirements-colab.txt\n    #   acvl-utils\n    #   batchgenerators\n    #   batchgeneratorsv2\n    #   blosc2\n    #   connected-components-3d\n    #   contourpy\n    #   dynamic-network-architectures\n    #   imagecodecs\n    #   imageio\n    #   matplotlib\n    #   nibabel\n    #   nnunetv2\n    #   numexpr\n    #   pandas\n    #   scikit-image\n    #   scikit-learn\n    #   scipy\n    #   seaborn\n    #   tifffile\n    #   torchvision\nnvidia-cublas-cu12==12.4.5.8\n    # via\n    #   nvidia-cudnn-cu12\n    #   nvidia-cusolver-cu12\n    #   torch\nnvidia-cuda-cupti-cu12==12.4.127\n    # via torch\nnvidia-cuda-nvrtc-cu12==12.4.127\n    # via torch\nnvidia-cuda-runtime-cu12==12.4.127\n    # via torch\nnvidia-cudnn-cu12==9.1.0.70\n    # via torch\nnvidia-cufft-cu12==11.2.1.3\n    # via torch\nnvidia-curand-cu12==10.3.5.147\n    # via torch\nnvidia-cusolver-cu12==11.6.1.9\n    # via torch\nnvidia-cusparse-cu12==12.3.1.170\n    # via\n    #   nvidia-cusolver-cu12\n    #   torch\nnvidia-cusparselt-cu12==0.6.2\n    # via torch\nnvidia-nccl-cu12==2.21.5\n    # via torch\nnvidia-nvjitlink-cu12==12.4.127\n    # via\n    #   nvidia-cusolver-cu12\n    #   nvidia-cusparse-cu12\n    #   torch\nnvidia-nvtx-cu12==12.4.127\n    # via torch\npackaging==26.3\n    # via\n    #   huggingface-hub\n    #   lazy-loader\n    #   matplotlib\n    #   nibabel\n    #   scikit-image\npandas==3.0.6\n    # via\n    #   batchgenerators\n    #   nnunetv2\n    #   seaborn\npillow==13.0.0\n    # via\n    #   batchgenerators\n    #   imageio\n    #   matplotlib\n    #   scikit-image\n    #   torchvision\npydantic==2.14.2\n    # via blosc2\npydantic-core==2.50.2\n    # via pydantic\npygments==2.21.0\n    # via rich\npyparsing==3.3.3\n    # via matplotlib\npython-dateutil==4.0.0\n    # via\n    #   matplotlib\n    #   pandas\npyyaml==6.0.3\n    # via\n    #   huggingface-hub\n    #   timm\n    #   yacs\nrequests==2.34.2\n    # via nnunetv2\nrich==15.0.0\n    # via blosc2\nsafetensors==0.8.0\n    # via timm\nscikit-image==0.26.0\n    # via\n    #   acvl-utils\n    #   batchgenerators\n    #   nnunetv2\nscikit-learn==1.9.1\n    # via\n    #   batchgenerators\n    #   nnunetv2\nscipy==1.17.1\n    # via\n    #   batchgenerators\n    #   nnunetv2\n    #   scikit-image\n    #   scikit-learn\nseaborn==0.13.2\n    # via nnunetv2\nsimpleitk==2.5.7\n    # via\n    #   acvl-utils\n    #   nnunetv2\nsympy==1.13.1\n    # via torch\nthreadpoolctl==3.7.0\n    # via\n    #   batchgenerators\n    #   blosc2\n    #   scikit-learn\ntifffile==2026.3.3\n    # via\n    #   nnunetv2\n    #   scikit-image\ntimm==1.0.22\n    # via dynamic-network-architectures\ntorch==2.6.0+cu124\n    # via\n    #   -r tools/e1_training/requirements-colab.txt\n    #   acvl-utils\n    #   batchgeneratorsv2\n    #   dynamic-network-architectures\n    #   nnunetv2\n    #   timm\n    #   torchvision\ntorchvision==0.21.0+cu124\n    # via timm\ntqdm==4.70.1\n    # via\n    #   acvl-utils\n    #   huggingface-hub\n    #   nnunetv2\ntriton==3.2.0\n    # via torch\ntruststore==0.10.5\n    # via\n    #   httpcore2\n    #   httpx2\ntyping-extensions==4.16.0\n    # via\n    #   anyio\n    #   httpx2\n    #   huggingface-hub\n    #   nibabel\n    #   pydantic\n    #   pydantic-core\n    #   torch\n    #   typing-inspection\ntyping-inspection==0.4.4\n    # via pydantic\ntzdata==2026.6\n    # via python-dateutil\nurllib3==2.8.0\n    # via requests\nyacs==0.1.8\n    # via nnunetv2\n'}
for name, source in EMBEDDED.items():
    (TOOLS / name).write_text(source)
sys.path.insert(0, str(TOOLS))
from colab_helpers import extract_handoff, sha256, require, run_logged
EXPECTED_DATASET_SHA256 = 'a3d7801021287c538ee8f3af55947b3795fe3ccf5d628f01d34942e0b4f93ae0'
EXPECTED_SPLIT_SHA256 = '9e2b37ffccff8196f9457963e3e603eabfeae6e89d8baec91ab5f989845d78b8'
EXPECTED_ARCHIVE_SHA256 = 'c580a1706a8242afc11b3e34c7b74586e759766e89fc4f3bd0c82cacb0be071a'
EXPECTED_CHECKPOINT_SHA256 = '678867420134d54b31a9975f2c2d1a34b57b8369aef5692fa64f97a8980a531b'


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.11.8"], check=True)
uv = [sys.executable, "-m", "uv"]
if not PYTHON.is_file():
    subprocess.run(uv + ["venv", str(VENV), "--python", "3.11"], check=True)
if RUN_LIVE:
    subprocess.run(uv + ["pip", "install", "--python", str(PYTHON), "--torch-backend", "cu124",
                         "-r", str(TOOLS / "requirements-colab.lock.txt")], check=True)
else:
    subprocess.run(uv + ["pip", "install", "--python", str(PYTHON),
                         "numpy==1.26.4", "nibabel==5.4.2", "matplotlib==3.11.2"], check=True)
subprocess.run(uv + ["pip", "install", "--python", str(PYTHON),
                     "scikit-image==0.26.0", "plotly==7.1.0"], check=True)
subprocess.run(uv + ["pip", "check", "--python", str(PYTHON)], check=True)
ENV = os.environ.copy()
ENV.update({"MPLBACKEND": "Agg", "MPLCONFIGDIR": str(WORK / "matplotlib"),
            "PYTHONNOUSERSITE": "1", "nnUNet_compile": "false",
            "OMP_NUM_THREADS": "4", "OPENBLAS_NUM_THREADS": "1", "MKL_NUM_THREADS": "1"})
print("Ortam hazır. Yeni tahmin seçimi:", RUN_LIVE)


## 3 · Deneyi değiştirmeden doğrulama vakasını seç
Sabit veri paketinin sağlama toplamı kontrol edilir. Test kümesi açılmaz.
Eğitim ön işlemesi yapılmaz; yeni tahminin kendi ön işlemesi tahmin adımındadır.


In [ ]:
require(sha256(ARCHIVE) == EXPECTED_ARCHIVE_SHA256, "ZIP, hazırlanan E1 arşiviyle aynı değil.")
extract_handoff(ARCHIVE, DATA)
require(sha256(DATA / "splits_final.json") == EXPECTED_SPLIT_SHA256, "Sabit ayrım değişmiş.")
split = json.loads((DATA / "splits_final.json").read_text())[0]
require(CASE_ID in split["val"] and CASE_ID not in split["train"], "Bir doğrulama vakası seç.")
manifest = json.loads((DATA / "expected_files.json").read_text())
require(manifest["dataset_sha256"] == EXPECTED_DATASET_SHA256, "Başka veri paketi seçilmiş.")
MRI = DATA / "nnUNet_raw" / DATASET / "imagesTr" / f"{CASE_ID}_0000.nii.gz"
REFERENCE = DATA / "nnUNet_raw" / DATASET / "labelsTr" / f"{CASE_ID}.nii.gz"
for path in (MRI, REFERENCE):
    require(sha256(path) == manifest["files"][path.relative_to(DATA).as_posix()], "Vaka dosyası değişmiş.")
RECORDED = SAVED / f"{CASE_ID}.nii.gz"
require(RECORDED.is_file(), f"Kaydedilmiş doğrulama tahmini yok: {RECORDED}")
print(f"Seçilen vaka: {CASE_ID} — doğrulama, bağımsız test değil.")


## 4 · Kaydedilmiş sonucu göster — GPU gerekmez
Bu hücre yeni tahmin üretmez. Önceden hesaplanan segmentasyonu MRI ve referansla
karşılaştırır; vaka Dice skorunu dosyalardan yeniden hesaplar.
Raporun başında **döndürülebilir 3B yüzey görünümü** bulunur: dokuları seç,
görünürlüklerini değiştir, Beyin/Kafa/Tümü görünümlerini dene. Altında 2B karşılaştırma korunur.
Bu yüzeyler simülasyon için hazırlanmış SimNIBS mesh'i değildir.


In [ ]:
RECORDED_REPORT = OUTPUTS / f"{CASE_ID}_recorded_{time.time_ns()}"
run_logged([str(PYTHON), str(TOOLS / "demo.py"), "report", "--mri", str(MRI),
            "--reference", str(REFERENCE), "--recorded-prediction", str(RECORDED),
            "--summary", str(SUMMARY), "--out", str(RECORDED_REPORT)],
           WORK / f"report_{time.time_ns()}.log", ENV)
from IPython.display import HTML, display
display(HTML((RECORDED_REPORT / "report.html").read_text()))
print("Sunum yedeği — internetsiz açılır:", RECORDED_REPORT / "report.html")


## 5 · Eğitilmiş modelle yeni tahmin — açık seçim
İlk hücrede `RUN_LIVE = True` yapıp ortam hücresini yeniden çalıştır.
GPU için çalışma zamanı türünü L4 seç; çalışma zamanı değişmişse 1–4'ü yeniden çalıştır.
Ağırlıklar yalnız okunur. Eksik/yanlış checkpoint durumunda kayıtlı tahmin canlıymış gibi gösterilmez.


In [ ]:
if RUN_LIVE:
    require(sha256(MODEL / "fold_0/checkpoint_final.pth") == EXPECTED_CHECKPOINT_SHA256,
            "Checkpoint, tamamlanan E1_L4_001 modelinden farklı.")
    LIVE_OUTPUT = OUTPUTS / f"{CASE_ID}_{DEVICE}_{time.time_ns()}"
    command = [str(PYTHON), str(TOOLS / "demo.py"), "predict", "--model", str(MODEL),
               "--mri", str(MRI), "--reference", str(REFERENCE), "--summary", str(SUMMARY),
               "--out", str(LIVE_OUTPUT), "--device", DEVICE]
    if not USE_TTA:
        command.append("--disable-tta")
    run_logged(command, WORK / f"inference_{time.time_ns()}.log", ENV)
    display(HTML((LIVE_OUTPUT / "report/report.html").read_text()))
    print("Yeni tahmin:", LIVE_OUTPUT / "prediction.nii.gz")
    print("Ölçülen süreler:", (LIVE_OUTPUT / "status.json").read_text())
else:
    print("Yeni tahmin kapalı. 4. bölümdeki çıktı, kaydedilmiş doğrulama tahminidir.")


## Sunum sırası
1. MRI girdisini ve renkli doku tahminini göster.
2. Referans/tahmin farklarını ve seçilen vaka skorlarını açıkla.
3. Kohort ortalamasının 10 doğrulama vakasına ait olduğunu belirt; test sonucu deme.
4. Canlı tahmin yaptıysan ölçülen cihazı ve ayna çevirmeli tahmin seçimini söyle.
5. Sonraki iş: etiket teyidi, diğer vakaların incelemesi, E2 verisi ve kontrollü karşılaştırmalar.

Sunumdan önce `report.html` ve `report.json` dosyalarını indir. GPU oturumunu işin bitince sonlandır.
Referans doku eşlemesi, yüzü silinmiş bölgeler ve fiziksel birim sınırlamaları geçerliliğini korur.
